# Agentic Pattern - Parallelization
Parallelization involves executing multiple components, such as LLM calls, tool usages, or even entire sub-agents, concurrently 

[![Agentic Pattern - Parallelization presentation](https://img.youtube.com/vi/?/0.jpg)](https://youtu.be/?) 

In [1]:
%%time
import warnings
warnings.filterwarnings("ignore")

!python3 -m pip install --upgrade pip

CPU times: user 8.03 ms, sys: 10.3 ms, total: 18.4 ms
Wall time: 1.04 s


In [2]:
%pip install -U -q langchain langchain-ollama ipython-autotime --use-deprecated=legacy-resolver

%load_ext autotime

Note: you may need to restart the kernel to use updated packages.
time: 125 μs (started: 2025-09-17 20:05:54 -07:00)


## Variables

In [3]:
my_model_ollama = "llama3.2"

time: 266 μs (started: 2025-09-17 20:05:54 -07:00)


## Initialize OLLAM service
OLLAMA inference client is initialized to interact with the OLLAMA API for generating responses from the specified model.

In [4]:
from langchain_ollama.llms import OllamaLLM

llm_client = OllamaLLM(
    model=my_model_ollama,
    base_url="http://localhost:11434",
    headers={"Content-Type": "application/json"},
    stream=True,
    temperature=0.7,
)

llm_client

OllamaLLM(model='llama3.2', temperature=0.7, base_url='http://localhost:11434')

time: 1.46 s (started: 2025-09-17 20:05:54 -07:00)


### TEST llm_client with a simple prompt

In [5]:
response = llm_client.invoke("What is Agentic Pattern - Parallelization")

print(f"{response}")

Agentic pattern - parallelization refers to a design approach in software development that aims to achieve concurrency and scalability by breaking down complex tasks into smaller, independent components that can be executed simultaneously.

In an agentic system, each component or process is designed to be self-contained, autonomous, and loosely coupled. This allows them to operate independently, making decisions based on local inputs and communicating with other components only when necessary.

The key characteristics of agentic systems are:

1. **Autonomy**: Each component has the freedom to make its own decisions and adapt to changing conditions.
2. **Loose Coupling**: Components interact with each other through well-defined interfaces, reducing dependencies and improving flexibility.
3. **Decentralization**: There is no central authority controlling the system; instead, components work together to achieve a common goal.

Parallelization in agentic systems involves creating multiple 

## Define Independent Chains
These three chains represent distinct tasks that can be executed in parallel.

In [6]:
from langchain_core.runnables import Runnable
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

summarize_chain: Runnable = (
    ChatPromptTemplate.from_messages(
        [("system", "Summarize the following topic concisely:"), ("user", "{topic}")]
    )
    | llm_client
    | StrOutputParser()
)

questions_chain: Runnable = (
    ChatPromptTemplate.from_messages(
        [
            (
                "system",
                "Generate three interesting questions about the following topic:",
            ),
            ("user", "{topic}"),
        ]
    )
    | llm_client
    | StrOutputParser()
)

terms_chain: Runnable = (
    ChatPromptTemplate.from_messages(
        [
            (
                "system",
                "Identify 5-10 key terms from the following topic, separated by commas:",
            ),
            ("user", "{topic}"),
        ]
    )
    | llm_client
    | StrOutputParser()
)

time: 29.3 ms (started: 2025-09-17 20:06:04 -07:00)


## Parallel Execution
RunnableParallel will run all tasks concurrently

1. Define the block of tasks to run in parallel. The results of these, along with the original topic, will be fed into the next step.

In [7]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough

parallel_tasks = RunnableParallel(
    {
        "summary": summarize_chain,
        "questions": questions_chain,
        "key_terms": terms_chain,
        "topic": RunnablePassthrough(),  # Pass the original topic
    }
)

parallel_tasks

{
  summary: ChatPromptTemplate(input_variables=['topic'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='Summarize the following topic concisely:'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['topic'], input_types={}, partial_variables={}, template='{topic}'), additional_kwargs={})])
           | OllamaLLM(model='llama3.2', temperature=0.7, base_url='http://localhost:11434')
           | StrOutputParser(),
  questions: ChatPromptTemplate(input_variables=['topic'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='Generate three interesting questions about the following topic:'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['topic'], input_types={}, partial_variables

time: 2.41 ms (started: 2025-09-17 20:06:04 -07:00)


2. Define the final synthesis prompt which will combine the parallel results.

In [8]:
synthesis_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """Based on the following information:
     Summary: {summary}
     Related Questions: {questions}
     Key Terms: {key_terms}
     Write a cohesive final answer that combines all three views in a clear way.""",
        ),
        ("user", "Original topic: {topic}"),
    ]
)

synthesis_prompt

ChatPromptTemplate(input_variables=['key_terms', 'questions', 'summary', 'topic'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['key_terms', 'questions', 'summary'], input_types={}, partial_variables={}, template='Based on the following information:\n     Summary: {summary}\n     Related Questions: {questions}\n     Key Terms: {key_terms}\n     Write a cohesive final answer that combines all three views in a clear way.'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['topic'], input_types={}, partial_variables={}, template='Original topic: {topic}'), additional_kwargs={})])

time: 1.77 ms (started: 2025-09-17 20:06:04 -07:00)


In [9]:
synthesis_chain = synthesis_prompt | llm_client | StrOutputParser()

time: 538 μs (started: 2025-09-17 20:06:04 -07:00)


 3. Construct the full chain by piping the parallel results directly into the synthesis prompt, followed by the LLM and output parser.


In [10]:
full_chain = parallel_tasks | synthesis_chain

full_chain

{
  summary: ChatPromptTemplate(input_variables=['topic'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='Summarize the following topic concisely:'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['topic'], input_types={}, partial_variables={}, template='{topic}'), additional_kwargs={})])
           | OllamaLLM(model='llama3.2', temperature=0.7, base_url='http://localhost:11434')
           | StrOutputParser(),
  questions: ChatPromptTemplate(input_variables=['topic'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='Generate three interesting questions about the following topic:'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['topic'], input_types={}, partial_variables

time: 1.71 ms (started: 2025-09-17 20:06:04 -07:00)


## Run the Chain
 Asynchronously invokes the parallel processing chain with a specific topic and prints the synthesized result.

In [11]:
my_topic = "The history of space exploration"

result = full_chain.invoke(my_topic)

print(f"\n--- Running Parallelization Pattern and Result --- \n")
print(f"{result}")


--- Running Parallelization Pattern and Result --- 

The History of Space Exploration: A Comprehensive Overview

The history of space exploration is a rich and fascinating narrative that spans over six decades. From the early years of artificial satellites to the current developments in private spaceflight, human space travel has made tremendous progress.

In the 1950s and 1960s, the Soviet Union launched Sputnik 1, the world's first artificial satellite, which marked a significant milestone in the history of space exploration. The United States responded with Explorer 1, launched by NASA in 1958, and subsequently achieved several historic achievements, including landing astronauts on the Moon.

The Apollo 11 mission, launched in 1969, successfully landed astronauts Neil Armstrong and Edwin "Buzz" Aldrin on the surface of the Moon, marking a historic achievement for human spaceflight. This achievement was made possible by the development of the Saturn V rocket, which became one of the